# Prometheus – LLM Judge

In [ ]:

# PROMETHEUS – LLM AS A JUDGE
# --------------------------------------------------
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import torch, datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Unbabel/M-Prometheus-7B"
print("Loading Prometheus (4‑bit) …")
from transformers import BitsAndBytesConfig
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_cfg,
    device_map="auto",
    use_auth_token=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.eval()

prompt = '''###Task Description:
An instruction ... (see full template above)'''

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
start = datetime.datetime.now()
print("▶ Generating judgement ...")
with torch.no_grad():
    out = model.generate(**inputs, max_length=1024, do_sample=False)
print("⏱", datetime.datetime.now() - start)
print(tokenizer.decode(out[0], skip_special_tokens=True))
